# run_dbt — Install dbt and execute `dbt build` against a target

This notebook is designed to be triggered by a Databricks Job (see Part 18).
The `target` parameter controls which `profiles.yml` output is used:
`dev`, `ci`, or `prod`.

**Prerequisites:**
- Databricks Secrets scope `aw` populated with `host`, `http_path`, `dbt_token`, `dbt_user`.
- The repo cloned as a Databricks Git folder at the path used in Cell 6.

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────
# When triggered by a Job, the Job passes `target` as a widget parameter.
# When run interactively, the default "dev" is used unless overridden.
dbutils.widgets.text("target", "dev")
target = dbutils.widgets.get("target") or "dev"
print(f"dbt target: {target}")

In [ ]:
# ── Install dbt ───────────────────────────────────────────────────────
# Pin to the same minor version used locally (see Part 3).
%pip install "dbt-databricks==1.12.*"

In [ ]:
# ── Restart Python after %pip install ─────────────────────────────────
# Required so the newly installed packages are importable in subsequent cells.
dbutils.library.restartPython()

In [ ]:
# ── Re-read the target widget after restart ───────────────────────────
# restartPython() clears all in-memory state, so we re-read the widget.
target = dbutils.widgets.get("target") or "dev"
print(f"dbt target (post-restart): {target}")

In [ ]:
# ── Inject secrets as environment variables ───────────────────────────
# Secrets are read from the 'aw' Databricks Secret scope.
# Values are NEVER printed — Databricks redacts them automatically.
# To create the scope, run from your local terminal (see Part 18 §3):
#   databricks secrets create-scope aw
#   databricks secrets put-secret aw host --string-value "adb-xxxx..."
#   databricks secrets put-secret aw http_path --string-value "/sql/1.0/..."
#   databricks secrets put-secret aw dbt_token --string-value "dapi..."
#   databricks secrets put-secret aw dbt_user --string-value "dapi..."
import os
os.environ["DBT_DBX_HOST"]           = dbutils.secrets.get(scope="aw", key="host")
os.environ["DBT_DBX_HTTP_PATH"]      = dbutils.secrets.get(scope="aw", key="http_path")
os.environ["DBT_DBX_HTTP_PATH_PROD"] = dbutils.secrets.get(scope="aw", key="http_path")
os.environ["DBT_DBX_TOKEN"]          = dbutils.secrets.get(scope="aw", key="dbt_token")
os.environ["DBT_USER"]               = dbutils.secrets.get(scope="aw", key="dbt_user")
os.environ["DBT_TARGET"]             = target
print("Secrets loaded. Running against target:", target)

In [ ]:
%sh
# ── Run dbt ───────────────────────────────────────────────────────────
# %sh MUST be the very first line of the cell — no Python comments above it,
# otherwise Databricks parses the cell as Python and `cd "$REPO"` raises
# SyntaxError: invalid syntax.
# %sh inherits the environment variables set in the previous Python cell.
# Adjust the REPO path to match your Git folder location in the workspace.
set -e   # exit immediately on any error — causes the Job task to fail cleanly

REPO="/Workspace/Users/hjtc365@gmail.com/adventureworks-databricks-medallion-dbt"
cd "$REPO"

# dbt checks the current working directory for profiles.yml before falling
# back to ~/.dbt/, so cd-ing into the repo root is sufficient — no
# --profiles-dir flag needed.

echo "==> dbt deps"
dbt deps

echo "==> dbt source freshness (target: $DBT_TARGET)"
dbt source freshness --target "$DBT_TARGET"

echo "==> dbt build (target: $DBT_TARGET)"
dbt build --target "$DBT_TARGET"

echo "==> Done."
